# Project: Dry Bean Type Classification

#### Tasks 1. Import and Load the Data
#### •	Import necessary libraries (pandas, numpy, matplotlib, seaborn, sklearn etc.)
#### •	Load the dataset and explore it using .head(), .info(), and .describe()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Create images directory if it doesn't exist
if not os.path.exists('images'):
    os.makedirs('images')

In [ ]:
try:
    df = pd.read_csv('Dry_Beans_Dataset.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: Dry_Beans_Dataset.csv not found.")

In [ ]:
display(df.head())
display(df.info())
display(df.describe())

#### 2. Exploratory Data Analysis (EDA)
#### •	Visualize distributions of features using histograms and boxplots
#### •	Analyze the class distribution (check for class imbalance)
#### •	Plot feature correlations (eg heatmap)
#### •	Visualize multivariate relationships (pairplot)
#### •	Summarize key findings

In [ ]:
# Class Distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='Class', data=df)
plt.title('Class Distribution')
plt.show()

print("Class Counts:")
print(df['Class'].value_counts())

In [ ]:
# Feature Distributions
numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numerical features: {numerical_features}")

# Histograms
df[numerical_features].hist(bins=20, figsize=(20, 15))
plt.suptitle('Histograms of Numerical Features')
plt.show()

In [ ]:
# Boxplots for outliers
plt.figure(figsize=(20, 10))
sns.boxplot(data=df[numerical_features])
plt.xticks(rotation=90)
plt.title('Boxplots of Numerical Features')
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 10))
correlation_matrix = df[numerical_features].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Pairplot
subset_features = ['Area', 'Perimeter', 'Compactness', 'roundness', 'Class']
sns.pairplot(df[subset_features], hue='Class')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

def summarize_eda(df):
    summary = {}

    # 1. Identify highly correlated numerical features
    numerical_df = df.select_dtypes(include=[np.number])

    # Calculate the correlation matrix for these numerical features.
    correlation_matrix = numerical_df.corr()

    # Define a threshold for what constitutes a 'high' correlation.
    high_correlation_threshold = 0.7

    # Filter the correlation matrix to find pairs of features with correlation
    highly_correlated_features = correlation_matrix[
        (correlation_matrix > high_correlation_threshold) & (correlation_matrix < 1.0)
    ]

    # Remove rows and columns that are entirely NaN after filtering.
    # This cleans up the output to only show actual high correlation pairs.
    highly_correlated_features = highly_correlated_features.dropna(how='all').dropna(axis=1, how='all')
    summary['high_correlations'] = highly_correlated_features

    # 2. Analyze Class Distribution
    # Get the count of each unique value in the 'Class' column.
    class_counts = df['Class'].value_counts()
    # Convert the series of counts to a dictionary for the summary output.
    summary['class_distribution'] = class_counts.to_dict()

    # 3. Basic Descriptive Statistics for Key Features
    # Define a specific subset of numerical features for which to calculate detailed statistics.
    subset_features_for_stats = ['Area', 'Perimeter', 'Compactness', 'roundness']
    # Calculate descriptive statistics (like mean, std, min, max, quartiles) for these features.
    summary['feature_stats'] = df[subset_features_for_stats].describe().to_dict()

    return summary

eda_summary = summarize_eda(df)

# Display the results of the EDA summary in a readable format using pprint.
import pprint
pprint.pprint(eda_summary)

#### 3. Missing Values & Outlier Treatment
#### •	Check for and handle missing values
#### •	Detect and treat outliers if needed (Z-score / IQR methods/boxplots)

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values[missing_values > 0])

if missing_values.sum() == 0:
    print("\nNo missing values found in the dataset.")
else:
    print("\nHandling missing values...")

In [ ]:
# Outlier Detection using IQR
def detect_outliers_iqr(data, features):
    outlier_indices = []
    for feature in features:
        Q1 = np.percentile(data[feature], 25)
        Q3 = np.percentile(data[feature], 75)
        IQR = Q3 - Q1
        outlier_step = 1.5 * IQR
        outliers_list_feature = data[(data[feature] < Q1 - outlier_step) | (data[feature] > Q3 + outlier_step)].index
        outlier_indices.extend(outliers_list_feature)
    
    return outlier_indices

outliers = detect_outliers_iqr(df, numerical_features)
print(f"Total number of outlier instances detected (with duplicates): {len(outliers)}")
print(f"Number of unique outlier instances: {len(set(outliers))}")

In [ ]:
# Outlier Treatment (removal of substantial outliers)
from collections import Counter

def remove_outliers(df, features, n=2):
    """
    Removes rows that contain outliers in n or more columns.
    """
    outlier_indices = []
    for feature in features:
        Q1 = np.percentile(df[feature], 25)
        Q3 = np.percentile(df[feature], 75)
        IQR = Q3 - Q1
        outlier_step = 1.5 * IQR
        outlier_list_col = df[(df[feature] < Q1 - outlier_step) | (df[feature] > Q3 + outlier_step)].index
        outlier_indices.extend(outlier_list_col)
        
    outlier_indices = Counter(outlier_indices)
    multiple_outliers = list(k for k, v in outlier_indices.items() if v > n)
    
    return multiple_outliers

outliers_to_drop = remove_outliers(df, numerical_features, n=2)
print(f"Dropping {len(outliers_to_drop)} observations with outliers in > 2 features.")

df_clean = df.drop(outliers_to_drop, axis=0).reset_index(drop=True)
print(f"Original shape: {df.shape}")
print(f"New shape: {df_clean.shape}")

#### 4. Feature Engineering & Preprocessing
#### •	Scale numerical features (StandardScaler / MinMaxScaler)
#### •	Encode categorical variables if necessary
#### •	Check and treat skewness if required
#### •	Split data into train/test sets (use stratified sampling while splitting)

In [ ]:
# Encoding Categorical Variables
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_clean['Class'] = le.fit_transform(df_clean['Class'])
print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
# Check and Treat Skewness

X = df_clean.drop('Class', axis=1)
y = df_clean['Class']

skewness = X.skew()
print("Skewness before treatment:\n", skewness)

# Treating Skewness (Log transformation for highly skewed features > 1)
skewed_features = skewness[abs(skewness) > 1].index
print(f"\nFeatures to treat for skewness: {skewed_features.tolist()}")

if len(skewed_features) > 0:
    for feature in skewed_features:
        # Use log1p (log(1+x)) to avoid log(0) and handle positive skewed data better
        X[feature] = np.log1p(X[feature])
    
    print("\nSkewness after treatment:\n", X[skewed_features].skew())

In [ ]:
# Splitting Data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

In [ ]:
# Scaling Numerical Features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# Fit on training set only
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for clearer viewing
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("Data Scaling Completed. First 5 rows of X_train_scaled:")
display(X_train_scaled.head())

#### 5. Model Building: Try Multiple Classifiers
#### Train and test the dataset on a variety of supervised classification algorithms:
#### •	Logistic Regression
#### •	Decision Tree Classifier
#### •	Random Forest Classifier
#### •	K-Nearest Neighbors (KNN)
#### •	Support Vector Machine (SVM)
#### •	Ensemble Learning Methods
#### •	Naive Bayes and more…
#### Use Cross validation techniques to check if model performance is improved.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Initialize models
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(random_state=42),
    "Naive Bayes": GaussianNB(),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

In [ ]:
results = []
names = []

print("Model Performance Evaluation (Cross-Validation & Test Set):")
print("-" * 80)
print(f"{'Model':<20} {'CV Mean Accuracy':<25} {'Test Accuracy':<15} {'F1-Score':<15}")
print("-" * 80)

for name, model in models.items():
    # Cross Validation (using Stratified K-Fold)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=skf, scoring='accuracy')
    
    # Train heavily on full train set
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    # Metrics
    test_acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results.append(cv_scores)
    names.append(name)
    
    print(f"{name:<20} {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})     {test_acc:.4f}           {f1:.4f}")

print("-" * 80)

In [ ]:
# Compare Algorithms
plt.figure(figsize=(12, 6))
plt.boxplot(results, labels=names)
plt.title('Algorithm Comparison (Cross-Validation Accuracy)')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.savefig('images/model_comparison.png')
plt.show()

#### 6. Handling Class Imbalance
#### •	Apply techniques like:
#####     o	SMOTE (Synthetic Minority Over-sampling)
#####     o	Random Oversampling / Undersampling
#####     o	Class weighting
#### •	Evaluate if performance improves on minority classes

In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# Visualize Class Imbalance in Training Set
print("Original Training Class Distribution:", Counter(y_train))
plt.figure(figsize=(10, 6))
y_train.value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title('Class Distribution in Training Set (Original)')
plt.xlabel('Class Label')
plt.ylabel('Count')
# plt.show()
plt.savefig('images/class_dist_original.png')

In [ ]:
# Define Resampling Methods
resampling_methods = {
    "SMOTE": SMOTE(random_state=42),
    "Random Oversampling": RandomOverSampler(random_state=42),
    "Random Undersampling": RandomUnderSampler(random_state=42)
}

# Define a Baseline Robust Model for Comparison (Random Forest)
rf_model = RandomForestClassifier(random_state=42)

# Baseline Performance (Imbalanced)
print("\n--- Baseline (Imbalanced) ---")
rf_model.fit(X_train_scaled, y_train)
y_pred_baseline = rf_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred_baseline))

In [ ]:
# Evaluate Resampling Techniques
for name, method in resampling_methods.items():
    print(f"\n--- Applying {name} ---")
    X_resampled, y_resampled = method.fit_resample(X_train_scaled, y_train)
    print(f"Resampled Training Class Distribution: {Counter(y_resampled)}")
    
    # Train Random Forest on Resampled Data
    rf_model.fit(X_resampled, y_resampled)
    y_pred_resampled = rf_model.predict(X_test_scaled)
    
    print(f"Performance with {name}:")
    print(classification_report(y_test, y_pred_resampled))

In [ ]:
# Evaluate Class Weighting
print("\n--- Applying Class Weighting (Balanced) ---")
rf_weighted = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_weighted.fit(X_train_scaled, y_train)
y_pred_weighted = rf_weighted.predict(X_test_scaled)
print("Performance with Class Weighting:")
print(classification_report(y_test, y_pred_weighted))

#### 7. Model Evaluation & Overfitting Check
#### Use appropriate classification metrics:
#### •	Accuracy
#### •	Precision, Recall, and F1-Score (for each class)
#### •	Confusion Matrix
#### Check for Overfitting
#### Compare training vs test accuracy and performance metrics.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Final Model Selection: Random Forest with Class Weighting
final_model = RandomForestClassifier(random_state=42, class_weight='balanced')

# Train on full Training Set (scaled)
final_model.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = final_model.predict(X_train_scaled)
y_test_pred = final_model.predict(X_test_scaled)

# Overfitting Check
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print("\n--- Overfitting Check ---")
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:     {test_acc:.4f}")
print(f"Gap:               {train_acc - test_acc:.4f}")

In [ ]:
# Detailed Classification Report
print("\n--- Detailed Classification Report (Test Set) ---")
print(classification_report(y_test, y_test_pred, target_names=le.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix (Random Forest - Balanced)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
# plt.show()
plt.savefig('images/confusion_matrix.png')

#### 8. Hyperparameter Tuning
#### •	Use GridSearchCV or RandomizedSearchCV to optimize parameters for top-performing models
#### •	Document the best parameters and performance improvement

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Tuning Random Forest
rf_params = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

rf = RandomForestClassifier(random_state=42, class_weight='balanced')

rf_random = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_params,
    n_iter=20, 
    cv=3,      
    verbose=2,
    random_state=42,
    n_jobs=-1   # Use all cores
)

print("\n--- Tuning Random Forest ---")
rf_random.fit(X_train_scaled, y_train)

print("Best Parameters for Random Forest:")
print(rf_random.best_params_)
print(f"Best CV Score: {rf_random.best_score_:.4f}")

In [ ]:
# Evaluate Best Random Forest on Test Set
best_rf = rf_random.best_estimator_
y_test_pred_tuned = best_rf.predict(X_test_scaled)

print("\n--- Tuned Random Forest Performance ---")
print(classification_report(y_test, y_test_pred_tuned))

In [ ]:
# 2. Tuning SVM
svm_params = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf', 'linear']  
}

svc = SVC(random_state=42, class_weight='balanced')

svm_random = RandomizedSearchCV(
    estimator=svc,
    param_distributions=svm_params,
    n_iter=10,  
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

print("\n--- Tuning SVM ---")
svm_random.fit(X_train_scaled, y_train)

print("Best Parameters for SVM:")
print(svm_random.best_params_)
print(f"Best CV Score: {svm_random.best_score_:.4f}")

In [ ]:
# Evaluate Best SVM
best_svm = svm_random.best_estimator_
y_test_pred_svm_tuned = best_svm.predict(X_test_scaled)

print("\n--- Tuned SVM Performance ---")
print(classification_report(y_test, y_test_pred_svm_tuned))

#### 9. Model Comparison Table
#### Model	Train Accuracy	Test Accuracy	F1 Score  	Overfitting (Y/N)
#### Logistic Regression				
#### Decision Tree				
#### Random Forest				
#### SVM				
#### KNN				
#### and other algos..				
#### Best Model				

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

models_comparison = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(random_state=42),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results_data = []

print("Generating Model Comparison Table...")
print("-" * 90)
print(f"{'Model':<20} {'Train Accuracy':<15} {'Test Accuracy':<15} {'F1 Score':<10} {'Overfitting (Y/N)':<18}")
print("-" * 90)

for name, model in models_comparison.items():
    # Train the model
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)
    
    # Calculate Metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    f1 = f1_score(y_test, y_test_pred, average='weighted')
    
    # Check for Overfitting (Threshold: 5% gap)
    overfitting = "Yes" if (train_acc - test_acc) > 0.05 else "No"
    
    results_data.append({
        "Model": name,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "F1 Score": f1,
        "Overfitting (Y/N)": overfitting
    })
    
    print(f"{name:<20} {train_acc:.4f}          {test_acc:.4f}          {f1:.4f}     {overfitting:<18}")

print("-" * 90)

# Find Best Model
best_model_row = sorted(results_data, key=lambda x: x['Test Accuracy'], reverse=True)[0]
print(f"Best Model: {best_model_row['Model']} (Test Acc: {best_model_row['Test Accuracy']:.4f})")

#### 10.  Build a Simple Classifier App
#### •	Use Streamlit to create a basic UI
#### •	Input physical measurements of a bean and get predicted class

In [ ]:
import joblib
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd
import numpy as np

print("Preparing final model and artifacts...")

final_scaler = StandardScaler()
X_final_scaled = final_scaler.fit_transform(X)

final_svm = SVC(kernel='rbf', gamma='auto', C=10, random_state=42, probability=True)
final_svm.fit(X_final_scaled, y)

joblib.dump(final_svm, 'svm_model.joblib')
joblib.dump(final_scaler, 'scaler.joblib')
joblib.dump(le, 'label_encoder.joblib')
joblib.dump(skewed_features.tolist(), 'skewed_features.joblib')

print("Successfully saved:")
print("- svm_model.joblib")
print("- scaler.joblib")
print("- label_encoder.joblib")
print("- skewed_features.joblib")

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# App configuration
st.set_page_config(page_title="Dry Bean Classifier", page_icon="🫘", layout="wide")

# Load artifacts
@st.cache_resource
def load_artifacts():
    model = joblib.load('svm_model.joblib')
    scaler = joblib.load('scaler.joblib')
    le = joblib.load('label_encoder.joblib')
    skewed_features = joblib.load('skewed_features.joblib')
    return model, scaler, le, skewed_features

model, scaler, le, skewed_features = load_artifacts()

st.title("🫘 Dry Bean Classification App")
st.markdown("""
Predict the type of dry bean based on its physical measurements. 
This app uses a Support Vector Machine (SVM) model trained on the Dry Bean Dataset.
""")

# Input Area
st.sidebar.header("Bean Measurements")

def user_input_features():
    # List of all features based on the dataset
    feature_names = [
        'Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 
        'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 
        'Extent', 'Solidity', 'roundness', 'Compactness', 
        'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4'
    ]
    
    # Default values (roughly based on dataset mean/median for SIRA or similar)
    defaults = {
        'Area': 10000.0, 'Perimeter': 500.0, 'MajorAxisLength': 150.0, 
        'MinorAxisLength': 100.0, 'AspectRation': 1.5, 'Eccentricity': 0.7, 
        'ConvexArea': 11000.0, 'EquivDiameter': 120.0, 'Extent': 0.75, 
        'Solidity': 0.99, 'roundness': 0.85, 'Compactness': 0.8, 
        'ShapeFactor1': 0.005, 'ShapeFactor2': 0.001, 'ShapeFactor3': 0.6, 
        'ShapeFactor4': 0.99
    }
    
    inputs = {}
    
    # Use 2 columns for inputs in sidebar
    col1, col2 = st.sidebar.columns(2)
    
    for i, feature in enumerate(feature_names):
        col = col1 if i % 2 == 0 else col2
        inputs[feature] = col.number_input(feature, value=defaults[feature], format="%.6f")
        
    return pd.DataFrame(inputs, index=[0])

input_df = user_input_features()

# Display Input Samples
st.subheader("User Input Parameters")
st.write(input_df)

# Prediction Logic
if st.button("Predict Bean Type"):
    # Apply Preprocessing
    processed_df = input_df.copy()
    
    # log1p transformation for skewed features
    for feature in skewed_features:
        processed_df[feature] = np.log1p(processed_df[feature])
    
    # Scaling
    X_scaled = scaler.transform(processed_df)
    
    # Predict
    prediction = model.predict(X_scaled)
    prediction_proba = model.predict_proba(X_scaled)
    
    class_name = le.inverse_transform(prediction)[0]
    
    # Display Result
    st.success(f"### Predicted Class: **{class_name}**")
    
    # Display Probabilities
    st.subheader("Prediction Probabilities")
    prob_df = pd.DataFrame(prediction_proba, columns=le.classes_)
    st.bar_chart(prob_df.T)
    st.write(prob_df)

st.divider()
st.info("Note: Ensure the units of measurements match those used in the original project dataset.")

#### Run the Streamlit App
#### To run the app, go to the root directory and execute the following command in your terminal:
```bash
streamlit run app.py
```